# Resonance Lattice on Fabric — UDF pattern

One `.rlat` lives in OneLake behind a User Data Function. Every team member's notebook queries it over HTTPS — no per-user encoder install, no per-notebook download.

Writes two Delta tables per run: `udf_telemetry` (per query) and `udf_hits` (per retrieved passage). The companion notebook `fabric_analytics.ipynb` reads both and deploys a Power BI semantic model. Setup: [docs/user/FABRIC.md](../../docs/user/FABRIC.md).

In [ ]:
# All deps ship in the Fabric pure-python kernel.
import polars as pl
import requests
import notebookutils


## Configuration

Set `UDF_BASE_URL` to the URL from your UDF's *Generate invocation code* button, stripped of any trailing `/functions/<name>/invoke`.

In [ ]:
UDF_BASE_URL = (
    "https://2da36c9357b24b9ea853ce08251ae0b9.z2d.userdatafunctions"
    ".fabric.microsoft.com/v1/workspaces/2da36c93-57b2-4b9e-a853-ce08251ae0b9"
    "/userDataFunctions/47c7a93e-5d6a-47e0-8f5c-2cefa5cb08b3"
)
KM_NAME = "rlat-self-test"
DEMO_QUERIES = [
    "how do I configure storage modes",
    "what is verified retrieval drift status",
    "how does the encoder cache work",
]
TOP_K = 5
VERIFIED_ONLY = True


In [ ]:
import hashlib
import time
from datetime import datetime, timezone

# UDF invocation needs the Power BI scope, not the Fabric API scope
# (which is for item CRUD).
TOKEN = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def invoke_udf(function_name: str, parameters: dict | None = None):
    """POST and unwrap the {functionName, status, output, errors} envelope."""
    r = requests.post(
        f"{UDF_BASE_URL}/functions/{function_name}/invoke",
        headers=HEADERS, json=parameters or {},
    )
    r.raise_for_status()
    body = r.json()
    if body.get("status") and body["status"] != "Succeeded":
        raise RuntimeError(f"{function_name} -> {body['status']}: {body.get('errors')}")
    return body.get("output", body)


## Discovery

`list_kms` reads `Files/rlat/_manifest.json` — a maintainer-managed list, because the Fabric DataLake SDK doesn't expose directory listing.

In [ ]:
pl.DataFrame(invoke_udf("list_kms"))


## Search

`verifiedOnly=True` drops hits whose source bytes have drifted since build.

In [ ]:
telemetry: list[dict] = []
all_hits: list[dict] = []

for q in DEMO_QUERIES:
    t0 = time.perf_counter()
    out = invoke_udf("search", {
        "kmName": KM_NAME, "query": q,
        "topK": TOP_K, "verifiedOnly": VERIFIED_ONLY,
    })
    latency_ms = int((time.perf_counter() - t0) * 1000)
    hits = out.get("hits", [])
    cold = bool(out.get("cold", False))
    band = out.get("band", "?")
    executed_utc = datetime.now(timezone.utc)
    date_key = executed_utc.strftime("%Y-%m-%d")
    query_hash = hashlib.sha1(q.encode("utf-8")).hexdigest()[:16]

    n_hits = len(hits)
    scores = [h["score"] for h in hits]
    sources = {h["source_file"] for h in hits}
    drift_count = sum(1 for h in hits if h["drift_status"] != "verified")
    telemetry.append({
        "query":            q,
        "query_hash":       query_hash,
        "km_name":          KM_NAME,
        "n_hits":           n_hits,
        "top1_score":       scores[0] if scores else 0.0,
        "top1_top2_gap":    scores[0] - scores[1] if len(scores) >= 2 else (1.0 if scores else 0.0),
        "source_diversity": (len(sources) / n_hits) if n_hits else 0.0,
        "drift_fraction":   (drift_count / n_hits) if n_hits else 0.0,
        "latency_ms":       latency_ms,
        "band":             band,
        "cold":             cold,
        "refused":          n_hits == 0,
        "executed_utc":     executed_utc,
        "date_key":         date_key,
    })
    for rank, h in enumerate(hits, 1):
        all_hits.append({
            "query":        q,
            "query_hash":   query_hash,
            "km_name":      KM_NAME,
            "executed_utc": executed_utc,
            "date_key":     date_key,
            "rank":         rank,
            **h,
        })

pl.DataFrame(telemetry).select([
    "query", "n_hits", "cold", "refused",
    "top1_score", "top1_top2_gap",
    "drift_fraction", "latency_ms",
])


## Top hits

Verified passage bodies — what an LLM consumer sees as grounding context.

In [ ]:
pl.DataFrame(all_hits).select([
    "query", "rank", "score", "source_file", "drift_status",
    pl.col("text").str.slice(0, 200).alias("text_preview"),
]).head(15)


## Telemetry to Delta

Two append-only Delta tables: `udf_telemetry` (one row per UDF call) and `udf_hits` (one row per retrieved passage, FK = `query_hash`). The analytics notebook reads both. Schema-enabled lakehouses route under `Tables/<schema>/<table>`; schema-less under `Tables/<table>` — `getWithProperties` returns `defaultSchema` as the routing signal.

In [ ]:
SCHEMA = "rlat"
FACT_QUERY = "udf_telemetry"
FACT_HIT = "udf_hits"

# `getWithProperties` returns extended metadata including `defaultSchema`;
# the basic `get()` omits it.
lh = notebookutils.lakehouse.getWithProperties(
    notebookutils.runtime.context["defaultLakehouseName"]
)
tables_root = f"{lh['properties']['abfsPath']}/Tables"
storage_options = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}


def _resolve(name: str) -> tuple[str, str]:
    if lh["properties"].get("defaultSchema"):
        return f"{tables_root}/{SCHEMA}/{name}", f"{SCHEMA}.{name}"
    return f"{tables_root}/{name}", name


fq_path, fq_pretty = _resolve(FACT_QUERY)
fh_path, fh_pretty = _resolve(FACT_HIT)

# delta-rs auto-creates on first append; schema_mode="merge" tolerates
# additive column changes without breaking history.
pl.DataFrame(telemetry, schema={
    "query":            pl.Utf8,
    "query_hash":       pl.Utf8,
    "km_name":          pl.Utf8,
    "n_hits":           pl.Int64,
    "top1_score":       pl.Float64,
    "top1_top2_gap":    pl.Float64,
    "source_diversity": pl.Float64,
    "drift_fraction":   pl.Float64,
    "latency_ms":       pl.Int32,
    "band":             pl.Utf8,
    "cold":             pl.Boolean,
    "refused":          pl.Boolean,
    "executed_utc":     pl.Datetime("us", time_zone="UTC"),
    "date_key":         pl.Utf8,
}).write_delta(
    fq_path, mode="append",
    storage_options=storage_options,
    delta_write_options={"schema_mode": "merge"},
)
print(f"appended {len(telemetry)} row(s) to {fq_pretty}")

if all_hits:
    fact_hits = [{
        "query_hash":   h["query_hash"],
        "km_name":      h["km_name"],
        "executed_utc": h["executed_utc"],
        "date_key":     h["date_key"],
        "rank":         h["rank"],
        "passage_idx":  h["passage_idx"],
        "source_file":  h["source_file"],
        "score":        float(h["score"]),
        "drift_status": h["drift_status"],
        "char_length":  h["char_length"],
    } for h in all_hits]
    pl.DataFrame(fact_hits, schema={
        "query_hash":   pl.Utf8,
        "km_name":      pl.Utf8,
        "executed_utc": pl.Datetime("us", time_zone="UTC"),
        "date_key":     pl.Utf8,
        "rank":         pl.Int32,
        "passage_idx":  pl.Int64,
        "source_file":  pl.Utf8,
        "score":        pl.Float64,
        "drift_status": pl.Utf8,
        "char_length":  pl.Int64,
    }).write_delta(
        fh_path, mode="append",
        storage_options=storage_options,
        delta_write_options={"schema_mode": "merge"},
    )
    print(f"appended {len(fact_hits)} row(s) to {fh_pretty}")


## Outside Fabric

The same UDF answers `rlat search fabric://<alias>/<km> "..."` after `rlat fabric add <alias>=<udf-base-url>`. The CLI's `add` also scaffolds `.claude/skills/rlat-fabric-search/SKILL.md` so Claude Code lists the corpus as a registered tool. Setup details: [docs/user/FABRIC.md](../../docs/user/FABRIC.md).